In [2]:
import numpy as np
import pickle
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ogbench

In [3]:
# --- Config ---
DATASET_NAME = "visual-scene-play-v0"
SAVE_PATH = Path("../data/samples/ogbench_segments.pkl")  # where to save selected segments
CAMERA = "front_pixels"

In [ ]:
# --- Load dataset (compact keeps all obs including terminal frames) ---
_, (train_dataset, _) = ogbench.make_env_and_datasets(
    dataset_name=DATASET_NAME,
    dataset_only=True,
    compact_dataset=True,
)

observations = train_dataset["observations"]  # (N, H, W, C)
actions      = train_dataset["actions"]        # (N, 5)
terminals    = train_dataset["terminals"]      # (N,) 1.0 at episode end

# Split flat dataset into list of episodes
terminal_idxs = np.where(terminals == 1.0)[0]
episode_slices = []
start = 0
for end in terminal_idxs:
    episode_slices.append(slice(start, end + 1))
    start = end + 1

print(f"Loaded {len(episode_slices)} episodes, obs shape: {observations.shape}")

In [1]:
# --- Inspector state ---
saved_segments: list[dict] = []
pending_start: int | None = None

# --- Widgets ---
ep_slider    = widgets.IntSlider(value=0, min=0, max=len(episode_slices)-1, description="Episode", continuous_update=False, layout=widgets.Layout(width="600px"))
frame_slider = widgets.IntSlider(value=0, min=0, max=0, description="Frame",   continuous_update=False, layout=widgets.Layout(width="600px"))
mark_start_btn  = widgets.Button(description="Mark Start",   button_style="info")
mark_end_btn    = widgets.Button(description="Mark End & Add", button_style="success")
clear_btn       = widgets.Button(description="Clear Pending",  button_style="warning")
save_btn        = widgets.Button(description="Save All Segments", button_style="danger")
status_label    = widgets.Label(value="")
segments_output = widgets.Output()
img_output      = widgets.Output()

def current_episode():
    return episode_slices[ep_slider.value]

def current_obs():
    sl = current_episode()
    return observations[sl]

def render_frame(frame_idx: int):
    with img_output:
        clear_output(wait=True)
        ep_obs = current_obs()
        ep_len = len(ep_obs)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(ep_obs[frame_idx])
        ax.set_title(f"Episode {ep_slider.value} | Frame {frame_idx}/{ep_len-1}" +
                     (f" | Start marked: {pending_start}" if pending_start is not None else ""))
        # Highlight start marker if set
        if pending_start is not None:
            color = "green" if frame_idx >= pending_start else "red"
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(4)
        ax.axis("off")
        plt.tight_layout()
        plt.show()

def update_segments_view():
    with segments_output:
        clear_output(wait=True)
        if not saved_segments:
            print("No segments selected yet.")
            return
        print(f"{len(saved_segments)} segment(s) selected:")
        for i, seg in enumerate(saved_segments):
            print(f"  [{i}] episode={seg['episode']}  frames={seg['start']}–{seg['end']}  ({seg['end']-seg['start']+1} steps)")

def on_episode_change(change):
    global pending_start
    pending_start = None
    sl = current_episode()
    ep_len = len(observations[sl])
    frame_slider.max = ep_len - 1
    frame_slider.value = 0
    status_label.value = f"Episode {ep_slider.value}: {ep_len} frames"
    render_frame(0)

def on_frame_change(change):
    render_frame(frame_slider.value)

def on_mark_start(btn):
    global pending_start
    pending_start = frame_slider.value
    status_label.value = f"Start marked at frame {pending_start}. Now pick end frame and click 'Mark End & Add'."
    render_frame(frame_slider.value)

def on_mark_end(btn):
    global pending_start
    if pending_start is None:
        status_label.value = "⚠ Mark a start frame first!"
        return
    end = frame_slider.value
    if end <= pending_start:
        status_label.value = "⚠ End frame must be after start frame!"
        return
    sl = current_episode()
    ep_obs = observations[sl]
    ep_act = actions[sl]
    saved_segments.append({
        "episode": ep_slider.value,
        "start": pending_start,
        "end": end,
        "observations": ep_obs[pending_start:end+1].copy(),
        "actions": ep_act[pending_start:end+1].copy(),
    })
    status_label.value = f"✓ Segment added (ep={ep_slider.value}, frames {pending_start}–{end}). Total: {len(saved_segments)}"
    pending_start = None
    update_segments_view()
    render_frame(frame_slider.value)

def on_clear(btn):
    global pending_start
    pending_start = None
    status_label.value = "Pending start cleared."
    render_frame(frame_slider.value)

def on_save(btn):
    if not saved_segments:
        status_label.value = "⚠ No segments to save!"
        return
    SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    # Strip large arrays for summary, save full data
    with open(SAVE_PATH, "wb") as f:
        pickle.dump(saved_segments, f)
    status_label.value = f"✓ Saved {len(saved_segments)} segments to {SAVE_PATH}"

ep_slider.observe(on_episode_change, names="value")
frame_slider.observe(on_frame_change, names="value")
mark_start_btn.on_click(on_mark_start)
mark_end_btn.on_click(on_mark_end)
clear_btn.on_click(on_clear)
save_btn.on_click(on_save)

# --- Layout ---
controls = widgets.VBox([
    ep_slider,
    frame_slider,
    widgets.HBox([mark_start_btn, mark_end_btn, clear_btn, save_btn]),
    status_label,
])
display(widgets.HBox([widgets.VBox([controls, segments_output]), img_output]))

# Init
on_episode_change(None)
update_segments_view()

NameError: name 'widgets' is not defined

In [ ]:
# --- Load and inspect saved segments ---
with open(SAVE_PATH, "rb") as f:
    loaded = pickle.load(f)

print(f"{len(loaded)} segments loaded:")
for i, seg in enumerate(loaded):
    print(f"  [{i}] episode={seg['episode']}  frames={seg['start']}–{seg['end']}  obs={seg['observations'].shape}  actions={seg['actions'].shape}")